# DBSCAN — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.

DBSCAN unsupervised হলেও notebook-এর গঠন PointTransformerV2-এর সাথে হুবহু এক:
Config → Loader → Features → Metrics → MLflow → Algorithm → Parameter Grid →
Training (parameter search + early stopping + checkpoint/resume) → Visualization →
Inference helper → Testing → Final Inference।

PTv2-এর **epoch** = DBSCAN-এর **trial** (একটা eps + min_samples combination)।


In [44]:
# ── Install (run once if needed) ─────────────────────────────────────────────
# pip install laspy open3d numpy scikit-learn joblib tqdm mlflow matplotlib pandas

import os, glob, copy, random, logging
import numpy as np
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


In [45]:
# ── Configuration — change values here, nowhere else ────────────────────────
CONFIG = {
    "train_dir"      : "../data/train",   # labelled .las files
    "test_dir"       : "../data/test",    # unlabelled .las files for inference
    "checkpoint_dir" : "../checkpoints",  # where the best parameters are saved
    "num_classes"    : 2,                 # 0 = environment, 1 = wood powder
    "target_class"   : 1,                 # class we want to highlight (green)
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "seed"           : 42,
    # ── training (= parameter search) ──
    # PTv2-এর epoch loop-এর জায়গায় DBSCAN-এ (eps, min_samples) grid search চলে।
    # প্রতিটা combination = একটা trial. একই early stopping নীতি প্রযোজ্য।
    "patience"       : 8,                # এতগুলো trial-এ improvement না হলে থামবে
    "min_miou"       : 0.90,             # এই val mIoU পেলে সাথে সাথে থামবে
    # ── DBSCAN parameter grid ──
    # eps এখন সরাসরি মিটার এককে (clustering raw xyz-এ হয়, scaled space-এ না)
    "eps_values"         : [0.05, 0.08, 0.10, 0.15, 0.20, 0.30],
    "min_samples_values" : [10, 20, 40, 80],
    # ── speed: এত point-এ সরাসরি DBSCAN ধীর — voxel downsample করে cluster,
    #    তারপর প্রতিটা original point-এ nearest downsampled point-এর label ──
    "voxel_size"         : 0.05,     # metres; বড় করলে দ্রুত কিন্তু মোটা
    # ── plane removal (DBSCAN-এর আগে ground/wall বাদ) ──
    # pile, মেঝে, দেয়াল সংযুক্ত surface — plane আগে না সরালে DBSCAN সব
    # একসাথে এক cluster বানিয়ে ফেলে। RANSAC দিয়ে সমতল অংশ খুঁজে:
    #   wall   = খাড়া plane (normal-এর z-উপাদান ছোট)
    #   ground = অনুভূমিক plane যেটা cloud-এর নিচের দিকে
    "remove_planes"      : True,
    "max_planes"         : 4,        # সর্বোচ্চ কতগুলো plane খোঁজা হবে
    "plane_dist"         : 0.03,     # plane-এর কত কাছের point inlier (m)
    "wall_nz_max"        : 0.35,     # |normal_z| এর নিচে হলে = wall
    "ground_nz_min"      : 0.85,     # |normal_z| এর উপরে হলে = অনুভূমিক
    "ground_z_pct"       : 20,       # অনুভূমিক plane নিচের এই percentile-এ থাকলে = ground
    # ── cluster selection strategy ──
    # "largest" | "highest" | "densest"  — কোন cluster-টা wood powder ধরা হবে
    "selection"      : "largest",
    # ── feature space ──
    # "xyz" (শুধু coordinates) | "xyz+feat" (xyz + height + density)
    "feature_space"  : "xyz+feat",
    # ── visualization ──
    "max_vis_files"  : 3,                # how many test files to show in 3D
    # ── MLflow ──
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "DBSCAN_Segmentation",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
NUM_CLASSES = CONFIG["num_classes"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG["seed"])


In [46]:
# ── Point cloud loader ───────────────────────────────────────────────────────
def load_pointcloud(path):
    """Read a .las/.laz file. Returns (points[N,3], labels[N] or None)."""
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    return pts, labels

def list_files(folder):
    """Return sorted list of .las/.laz files in a folder."""
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

# ── Split labelled files into train / val / test ─────────────────────────────
all_files = list_files(CONFIG["train_dir"])
assert all_files, f"No .las files found in {CONFIG['train_dir']}"

rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_val  = max(1, int(len(all_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_files[i] for i in order[:n_val]]
TEST_FILES  = [all_files[i] for i in order[n_val : n_val + n_test]]
TRAIN_FILES = [all_files[i] for i in order[n_val + n_test :]]
INFER_FILES = list_files(CONFIG["test_dir"])   # no labels, final inference only

log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")


2026-07-10 05:27:47,744 | INFO | train=33  val=6  test=6  inference=18


In [47]:
# ── Feature computation (DBSCAN-এর ইনপুট) ───────────────────────────────────
# PTv2 যেমন 7-channel feature ব্যবহার করত, DBSCAN এখানে 3 বা 5-channel ব্যবহার করে।
def compute_features(pts):
    """DBSCAN clustering-এর জন্য feature vector।

    'xyz'      → normalize করা XYZ (3 channels)
    'xyz+feat' → XYZ + height_above_floor + local_point_density (5 channels)
                 height: pile উঁচুতে থাকে, environment নিচে
                 density: pile-এর surface ঘন, দেয়াল/মেঝে আলাদা density-তে
    """
    pts_min  = pts.min(axis=0)
    pts_max  = pts.max(axis=0)
    norm_xyz = (pts - pts_min) / np.clip(pts_max - pts_min, 1e-9, None)

    if CONFIG["feature_space"] == "xyz":
        return norm_xyz.astype(np.float32)

    # height above floor
    z      = pts[:, 2]
    height = (z - z.min()) / max(z.max() - z.min(), 1e-9)

    # local point density (kNN mean distance-এর inverse)
    k    = 16
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree",
                            n_jobs=-1).fit(pts)
    d, _ = nbrs.kneighbors(pts)
    mean_d  = d[:, 1:].mean(axis=1)              # skip self (distance 0)
    density = 1.0 / np.clip(mean_d, 1e-9, None)
    density = (density - density.min()) / max(density.max() - density.min(), 1e-9)

    return np.column_stack([norm_xyz, height, density]).astype(np.float32)


In [48]:
# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_miou(true_labels, pred_labels, num_classes):
    """Compute mean Intersection-over-Union across all classes."""
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0


def compute_dice(true_labels, pred_labels, num_classes):
    """Compute mean Dice Score across all classes."""
    dices = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            # Dice Formula: 2*TP / (2*TP + FP + FN)
            dices.append((2 * tp) / (2 * tp + fp + fn))
    return float(np.mean(dices)) if dices else 0.0


def compute_accuracy(true_labels, pred_labels):
    """Point-wise classification accuracy."""
    return float((true_labels == pred_labels).mean())


def compute_val_loss(true_labels, pred_labels, num_classes):
    """DBSCAN-এ gradient/loss নেই — তাই (1 - mIoU) কে val_loss হিসেবে
    ব্যবহার করা হয়। mIoU বাড়লে এটা কমে, PTv2-এর val_loss-এর মতোই আচরণ।"""
    return 1.0 - compute_miou(true_labels, pred_labels, num_classes)


In [49]:
# ── MLflow setup ─────────────────────────────────────────────────────────────
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")


2026-07-10 05:27:47,780 | WARNING | Retrying (Retry(total=6, connect=6, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=DBSCAN_Segmentation
2026-07-10 05:27:52,420 | WARNING | Retrying (Retry(total=5, connect=5, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=DBSCAN_Segmentation
2026-07-10 05:28:00,446 | WARNING | Retrying (Retry(total=4, connect=4, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [Errno 111] Connection refused")': /api/2.0/mlflow/experiments/get-by-name?e

## Algorithm

In [50]:
MODEL_NAME = "DBSCAN"

# Pipeline: voxel downsample → RANSAC plane removal (ground + wall) →
#           DBSCAN clustering → target cluster selection → label propagation
#
# কেন plane removal দরকার:
#   pile, মেঝে, দেয়াল শারীরিকভাবে সংযুক্ত surface — DBSCAN শুধু density আর
#   connectivity দেখে, তাই সরাসরি চালালে সব মিলে একটা cluster হয়ে যায়।
#   RANSAC দিয়ে আগে সমতল অংশ (মেঝে=অনুভূমিক, দেয়াল=খাড়া) সরিয়ে দিলে
#   pile আলাদা হয়ে যায়, তখন DBSCAN কাজ করতে পারে।

from scipy.spatial import cKDTree

def remove_planes(pts):
    """RANSAC দিয়ে ground ও wall plane সনাক্ত করে বাদ দেয়।

    প্রতিটা পাওয়া plane-কে normal দিয়ে শ্রেণীবদ্ধ করা হয়:
      wall   : |normal_z| < wall_nz_max        (খাড়া plane)
      ground : |normal_z| > ground_nz_min এবং plane-এর গড় z নিচের
               ground_z_pct percentile-এ  (অনুভূমিক ও নিচু)
      অন্য   : সম্ভবত pile-এর নিজের সমতল উপরিভাগ — environment ধরা হয় না,
               কিন্তু পরের plane খোঁজার সময় search থেকে বাদ থাকে

    Returns: env_mask[N] — True মানে সেই point ground/wall।"""
    n = len(pts)
    env_mask   = np.zeros(n, bool)
    search_idx = np.arange(n)          # এখনো search-এ থাকা point-দের index
    z_ground   = np.percentile(pts[:, 2], CONFIG["ground_z_pct"])

    for _ in range(CONFIG["max_planes"]):
        if len(search_idx) < 500:
            break
        work = pts[search_idx]
        pcd  = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(work)
        try:
            (a, b, c_, d), inliers = pcd.segment_plane(
                distance_threshold=CONFIG["plane_dist"],
                ransac_n=3, num_iterations=500)
        except Exception:
            break
        inliers = np.asarray(inliers)
        if len(inliers) < 0.03 * n:            # খুব ছোট plane → থামা
            break

        nz       = abs(c_) / max(np.linalg.norm([a, b, c_]), 1e-9)
        plane_z  = work[inliers, 2].mean()
        is_wall   = nz < CONFIG["wall_nz_max"]
        is_ground = (nz > CONFIG["ground_nz_min"]) and (plane_z <= z_ground)

        if is_wall or is_ground:
            env_mask[search_idx[inliers]] = True
            kind = "wall" if is_wall else "ground"
            log.info(f"  [plane] {kind}: {len(inliers):,} pts "
                     f"(|nz|={nz:.2f}, z̄={plane_z:.2f})")
        # plane-টা env হোক বা pile-এর উপরিভাগ — পরের iteration-এর search
        # থেকে বাদ, নইলে RANSAC বারবার একই plane খুঁজে পাবে
        keep       = np.ones(len(search_idx), bool)
        keep[inliers] = False
        search_idx = search_idx[keep]

    return env_mask


def select_target_cluster(pts, cluster_labels, method):
    """DBSCAN-এর অনেক cluster-এর মধ্যে কোনটা wood powder সেটা নির্বাচন করে।"""
    unique = [l for l in np.unique(cluster_labels) if l >= 0]
    if not unique:
        return -1

    if method == "largest":
        sizes = {l: (cluster_labels == l).sum() for l in unique}
        return max(sizes, key=sizes.get)
    elif method == "highest":
        means = {l: pts[cluster_labels == l, 2].mean() for l in unique}
        return max(means, key=means.get)
    elif method == "densest":
        scores = {}
        for l in unique:
            sub = pts[cluster_labels == l]
            rng = sub.max(0) - sub.min(0)
            vol = max(rng[0] * rng[1] * rng[2], 1e-9)
            scores[l] = len(sub) / vol
        return max(scores, key=scores.get)
    return unique[0]


def dbscan_segment(pts, eps, min_samples):
    """Full pipeline: downsample → plane removal → DBSCAN → selection →
    label propagation.  Returns (preds[N], cluster_labels[N], n_clusters)."""
    pts = np.asarray(pts, np.float64)

    # ── 1. voxel downsample (গতির জন্য) ─────────────────────────────────────
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    down = pcd.voxel_down_sample(voxel_size=CONFIG["voxel_size"])
    pts_down = np.asarray(down.points)
    if len(pts_down) < min_samples * 2:
        pts_down = pts

    # ── 2. ground/wall plane removal ─────────────────────────────────────────
    if CONFIG["remove_planes"]:
        env_down = remove_planes(pts_down)
    else:
        env_down = np.zeros(len(pts_down), bool)
    pile_candidates = pts_down[~env_down]

    # ── 3. DBSCAN on remaining points (raw metric xyz) ───────────────────────
    labels_down = np.full(len(pts_down), -1, dtype=np.int64)
    n_clusters  = 0
    if len(pile_candidates) >= min_samples:
        pcd_c = o3d.geometry.PointCloud()
        pcd_c.points = o3d.utility.Vector3dVector(pile_candidates)
        lab = np.asarray(pcd_c.cluster_dbscan(eps=eps, min_points=min_samples))
        labels_down[~env_down] = lab
        n_clusters = len(set(lab)) - (1 if -1 in lab else 0)

    # ── 4. target cluster selection ──────────────────────────────────────────
    target_cluster = select_target_cluster(pts_down, labels_down,
                                           CONFIG["selection"])
    target_down = (labels_down == target_cluster) if target_cluster >= 0                   else np.zeros(len(pts_down), bool)

    # ── 5. label propagation: original → nearest downsampled ────────────────
    tree = cKDTree(pts_down)
    _, nearest = tree.query(pts, k=1)
    preds = np.where(target_down[nearest], CONFIG["target_class"], 0).astype(np.int64)
    cluster_labels = labels_down[nearest]

    return preds, cluster_labels, n_clusters

log.info(f"{MODEL_NAME} ready — plane removal={'ON' if CONFIG['remove_planes'] else 'OFF'}, "
         f"voxel={CONFIG['voxel_size']}m, selection='{CONFIG['selection']}'")


2026-07-10 05:31:53,696 | INFO | DBSCAN ready — plane removal=ON, voxel=0.05m, selection='largest'


## Parameter Grid  (PTv2-এর Dataset & DataLoader-এর জায়গায়)

DBSCAN-এ Dataset/DataLoader লাগে না — তার জায়গায় parameter grid।
PTv2 যেমন epoch ধরে ধরে train করত, DBSCAN এই grid-এর প্রতিটা combination ধরে ধরে
চেষ্টা করবে (প্রতিটা combination = একটা trial)।

In [51]:
# ── Parameter grid (training-এর "data") ─────────────────────────────────────
# সব (eps, min_samples) combination-এর তালিকা তৈরি — training loop এটার
# উপর দিয়ে iterate করবে, PTv2 যেমন DataLoader-এর উপর দিয়ে করত।
#
# সতর্কতা: voxel downsample-এর পরে point-দের ন্যূনতম দূরত্ব ≈ voxel_size।
# তাই eps < 1.5×voxel_size হলে কেউ কারো প্রতিবেশী হবে না — সব noise।
# এমন অর্থহীন combination গুলো grid থেকে বাদ দেওয়া হয়।
_min_eps = 1.5 * CONFIG["voxel_size"]
PARAM_GRID = [(eps, ms)
              for eps in CONFIG["eps_values"] if eps >= _min_eps
              for ms  in CONFIG["min_samples_values"]]

_skipped = [e for e in CONFIG["eps_values"] if e < _min_eps]
if _skipped:
    log.warning(f"eps {_skipped} বাদ দেওয়া হলো (voxel_size={CONFIG['voxel_size']}m-এর "
                f"তুলনায় খুব ছোট, সব noise হয়ে যেত)")

log.info(f"Parameter grid: {len(PARAM_GRID)} combinations")


2026-07-10 05:31:53,714 | WARNING | eps [0.05] বাদ দেওয়া হলো (voxel_size=0.05m-এর তুলনায় খুব ছোট, সব noise হয়ে যেত)
2026-07-10 05:31:53,714 | INFO | Parameter grid: 20 combinations


## Training  (parameter search with early stopping + checkpoint/resume)

In [52]:
# ── Training loop (parameter search) with checkpointing ─────────────────────
# PTv2-এর train_model-এর হুবহু নীতি:
#   প্রতিটা trial-এ:     val set-এ mIoU/Dice/Acc/Loss হিসাব → MLflow-এ log
#   best হলে:            best parameters checkpoint-এ save (joblib)
#   early stopping:      patience trial-এ improvement না হলে বা min_miou পেলে থামে
#   resume policy:       checkpoint আগে থেকে থাকলে search পুরো skip

def train_model(model_name):
    """Search the parameter grid, save the best parameters, log to MLflow.
    Returns dict: {"eps": ..., "min_samples": ..., "best_miou": ...}"""

    ckpt_path = os.path.join(CONFIG["checkpoint_dir"], f"{model_name}_best.joblib")

    # ── resume: checkpoint থাকলে training পুরো skip ─────────────────────────
    # CKPT_VERSION: algorithm বদলালে (যেমন scaled-space → metric-space) পুরনো
    # checkpoint-এর parameter অর্থহীন হয়ে যায় — version না মিললে বাতিল।
    CKPT_VERSION = 3   # v3 = plane removal (ground+wall) + metric-space clustering
    if os.path.exists(ckpt_path):
        best = joblib.load(ckpt_path)
        ckpt_ok = (best.get("version") == CKPT_VERSION
                   and best.get("eps", 0) >= 1.5 * CONFIG["voxel_size"])
        if ckpt_ok:
            log.info(f"Checkpoint found at {ckpt_path} — loading, skipping search.")
            log.info(f"Loaded: eps={best['eps']}  min_samples={best['min_samples']}  "
                     f"val mIoU={best['best_miou']:.4f}")
            return best
        else:
            log.warning(f"পুরনো/অসঙ্গত checkpoint পাওয়া গেছে ({ckpt_path}) — "
                        f"বাতিল করে নতুন করে search চালানো হচ্ছে।")
            os.remove(ckpt_path)

    # ── validation ডেটা প্রস্তুত (সব VAL_FILES একসাথে) ───────────────────────
    # প্রতিটা val ফাইলে আলাদাভাবে segment করে গড় mIoU নেওয়া হয়
    val_data = []
    for path in VAL_FILES:
        pts, lbl = load_pointcloud(path)
        if lbl is not None:
            lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
            val_data.append((os.path.basename(path), pts, lbl))
    assert val_data, "Validation ফাইলে কোনো label নেই — search অসম্ভব।"

    best_miou  = -1.0
    best_eps   = None
    best_ms    = None
    no_improve = 0
    history    = []

    run_name = f"{model_name}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:

        if MLFLOW_OK:
            mlflow.log_params({
                "model"          : model_name,
                "trials_total"   : len(PARAM_GRID),
                "patience"       : CONFIG["patience"],
                "min_miou_target": CONFIG["min_miou"],
                "selection"      : CONFIG["selection"],
                "feature_space"  : CONFIG["feature_space"],
            })

        stop = False
        for trial, (eps, ms) in enumerate(PARAM_GRID, start=1):

            # ── এই trial-এর parameter দিয়ে সব val ফাইল segment ──────────────
            mious, dices, accs, losses = [], [], [], []
            for fname, pts, lbl in val_data:
                try:
                    preds, _, _ = dbscan_segment(pts, eps=eps, min_samples=ms)
                except Exception as e:
                    log.warning(f"  {fname}: eps={eps} ms={ms} ERROR {e}")
                    continue
                mious.append(compute_miou(lbl, preds, NUM_CLASSES))
                dices.append(compute_dice(lbl, preds, NUM_CLASSES))
                accs.append(compute_accuracy(lbl, preds))
                losses.append(compute_val_loss(lbl, preds, NUM_CLASSES))

            if not mious:
                continue

            miou     = float(np.mean(mious))
            dice     = float(np.mean(dices))
            acc      = float(np.mean(accs))
            val_loss = float(np.mean(losses))
            history.append({"trial": trial, "eps": eps, "min_samples": ms,
                            "val_miou": miou, "val_loss": val_loss})

            # ── Logging to Console (PTv2 format) ─────────────────────────────
            log.info(
                f"Trial {trial:3d}/{len(PARAM_GRID)} | "
                f"eps={eps:.3f} min_samples={ms:3d} | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {acc:.4f} | "
                f"mIoU: {miou:.4f} | Dice: {dice:.4f}"
            )

            # ── Logging to MLflow ─────────────────────────────────────────────
            if MLFLOW_OK:
                mlflow.log_metrics({
                    "val_loss": val_loss,
                    "val_acc" : acc,
                    "val_miou": miou,
                    "val_dice": dice,
                    "eps"         : eps,
                    "min_samples" : ms,
                }, step=trial)

            # ── update best ───────────────────────────────────────────────────
            if miou > best_miou + 1e-4:
                best_miou  = miou
                best_eps   = eps
                best_ms    = ms
                no_improve = 0
                best = {"version": CKPT_VERSION,
                        "eps": best_eps, "min_samples": best_ms,
                        "best_miou": best_miou,
                        "selection": CONFIG["selection"],
                        "feature_space": CONFIG["feature_space"],
                        "history": history}
                joblib.dump(best, ckpt_path)
                log.info(f"  ✓ New best mIoU {best_miou:.4f} → saved {ckpt_path}")
            else:
                no_improve += 1

            # ── early stopping ────────────────────────────────────────────────
            if miou >= CONFIG["min_miou"]:
                log.info(f"Early stop at trial {trial}: mIoU {miou:.4f} ≥ "
                         f"target {CONFIG['min_miou']}")
                stop = True
            if no_improve >= CONFIG["patience"]:
                log.info(f"Early stop at trial {trial} "
                         f"(no improvement for {CONFIG['patience']} trials)")
                stop = True
            if stop:
                break

        if MLFLOW_OK:
            mlflow.log_metric("best_val_miou", best_miou)

    best = joblib.load(ckpt_path)
    log.info(f"Best parameters: eps={best['eps']}  "
             f"min_samples={best['min_samples']}  (val mIoU={best['best_miou']:.4f})")
    return best


In [53]:
best_params = train_model(MODEL_NAME)

# best parameters CONFIG-এ বসানো — এরপর সব inference এগুলো ব্যবহার করবে
CONFIG["eps"]         = best_params["eps"]
CONFIG["min_samples"] = best_params["min_samples"]


2026-07-10 05:31:53,855 | INFO |   [plane] wall: 6,227 pts (|nz|=0.01, z̄=76.77)
2026-07-10 05:31:53,867 | INFO |   [plane] wall: 5,147 pts (|nz|=0.01, z̄=76.29)
2026-07-10 05:31:53,879 | INFO |   [plane] wall: 3,770 pts (|nz|=0.00, z̄=78.61)
2026-07-10 05:31:54,125 | INFO |   [plane] wall: 7,191 pts (|nz|=0.01, z̄=81.22)
2026-07-10 05:31:54,139 | INFO |   [plane] wall: 4,628 pts (|nz|=0.00, z̄=81.73)
2026-07-10 05:31:54,157 | INFO |   [plane] wall: 3,105 pts (|nz|=0.00, z̄=83.02)
2026-07-10 05:31:54,374 | INFO |   [plane] wall: 7,132 pts (|nz|=0.00, z̄=80.16)
2026-07-10 05:31:54,385 | INFO |   [plane] ground: 5,105 pts (|nz|=1.00, z̄=77.16)
2026-07-10 05:31:54,404 | INFO |   [plane] wall: 4,537 pts (|nz|=0.01, z̄=82.24)
2026-07-10 05:31:54,667 | INFO |   [plane] wall: 5,743 pts (|nz|=0.01, z̄=76.53)
2026-07-10 05:31:54,693 | INFO |   [plane] wall: 4,911 pts (|nz|=0.02, z̄=75.14)
2026-07-10 05:31:54,955 | INFO |   [plane] wall: 7,299 pts (|nz|=0.00, z̄=79.98)
2026-07-10 05:31:54,975 | 

## Visualization helper

In [54]:
# ── Visualization: Open3D ────────────────────────────────────────────────────
# Green = target (wood powder)   |   Red = others (environment)
def visualize_segmentation(points, predictions, title="Segmentation"):
    """Open an Open3D window showing the segmentation result."""
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]   # green
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]   # red

    pcd = o3d.geometry.PointCloud()
    # centre the cloud so it appears at the origin
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # XYZ axes
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)

    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)


## Inference helper

In [55]:
# ── Full-cloud inference ──────────────────────────────────────────────────────
def predict_full_cloud(path):
    """Label every point in a file using the best DBSCAN parameters.
    PTv2-এর predict_full_cloud-এর মতোই signature: returns (pts, preds, lbl)."""
    pts, lbl = load_pointcloud(path)
    if lbl is not None:
        lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
    preds, _, _ = dbscan_segment(pts,
                                 eps=CONFIG["eps"],
                                 min_samples=CONFIG["min_samples"])
    return pts, preds, lbl


## Volume Estimation (TIN)

Segmentation-এর পরে target points থেকে volume বের করা হয়।

**Pipeline (সব `vol_TIN()`-এর ভেতরে):**
1. **SOR** — noisy বিচ্ছিন্ন point বাদ (statistical distance filter)
2. **DBSCAN largest cluster** — stray mislabeled points বাদ
3. **TIN integration** — 2D Delaunay → Σ Area₂D(triangle) × mean height
   - Floor baseline = scanned z-এর `floor_pct` percentile (outer-shell scan-এ আলাদা floor point থাকে না)
   - কোনো alpha clipping নেই — DBSCAN-ই stray point সামলায়; clipping করলে কিনারার বৈধ triangle কেটে গিয়ে volume কমে যেত
4. **Bootstrap CI** — ৮০ বার subsample করে std + 95% confidence interval


In [56]:
# ── Volume Estimation: TIN-based volumetric integration ─────────────────────
from scipy.spatial import Delaunay, cKDTree


def sor_filter(pts, k=16, std_ratio=2.0):
    """Statistical Outlier Removal: mean-kNN-distance > mean + std_ratio*std
    হলে সেই point বাদ। Returns (filtered_pts, kept_mask)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) <= k + 1:
        return pts, np.ones(len(pts), bool)
    tree = cKDTree(pts)
    d, _ = tree.query(pts, k=k + 1)          # col-0 = নিজেই (দূরত্ব 0)
    mean_dist = d[:, 1:].mean(axis=1)
    thr  = mean_dist.mean() + std_ratio * mean_dist.std()
    mask = mean_dist <= thr
    return pts[mask], mask


def extract_main_cluster(pts, eps=0.15, min_samples=8):
    """DBSCAN দিয়ে সবচেয়ে বড় connected cluster রাখা — stray points বাদ।
    এর পরে আর alpha clipping লাগে না।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < min_samples * 2:
        return pts
    try:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        labels = np.asarray(pcd.cluster_dbscan(eps=eps, min_points=min_samples))
        valid = labels[labels >= 0]
        if valid.size == 0:
            return pts
        main_lbl = np.bincount(valid).argmax()
        result = pts[labels == main_lbl]
        return result if len(result) >= min_samples else pts
    except Exception as e:
        log.warning(f"[TIN] cluster extraction failed ({e}) -> using all points")
        return pts


def tin_integrate(pts, floor_pct=0.5):
    """Delaunay TIN integration.
    floor baseline = scanned z-এর floor_pct percentile (pile-এর base প্রান্ত)।
    Volume = Σ Area2D(triangle) × mean_height(triangle)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 4:
        return float("nan")

    floor_z = np.percentile(pts[:, 2], floor_pct)
    h  = pts[:, 2] - floor_z
    xy = pts[:, :2]
    try:
        tri = Delaunay(xy)
    except Exception as e:
        log.warning(f"[TIN] Delaunay failed ({e})")
        return float("nan")

    t = tri.simplices
    p0, p1, p2 = xy[t[:, 0]], xy[t[:, 1]], xy[t[:, 2]]
    h0, h1, h2 = h[t[:, 0]], h[t[:, 1]], h[t[:, 2]]

    cross  = ((p1[:, 0] - p0[:, 0]) * (p2[:, 1] - p0[:, 1])
             - (p1[:, 1] - p0[:, 1]) * (p2[:, 0] - p0[:, 0]))
    area2d = np.abs(cross) / 2.0
    mean_h = (h0 + h1 + h2) / 3.0
    valid  = mean_h > 0
    return float(np.sum(area2d[valid] * mean_h[valid]))


def bootstrap_tin_confidence(pts, n_bootstrap=80, subsample_frac=0.85,
                             floor_pct=0.5, seed=0):
    """Bootstrap: বারবার subsample করে TIN চালিয়ে uncertainty মাপা।
    Returns {'std', 'ci_low', 'ci_high'} বা None।"""
    rng = np.random.RandomState(seed)
    n = len(pts)
    n_sub = max(10, int(n * subsample_frac))
    estimates = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n_sub, replace=False)
        v = tin_integrate(pts[idx], floor_pct=floor_pct)
        if np.isfinite(v) and v > 0:
            estimates.append(v)
    if len(estimates) < 5:
        return None
    arr = np.array(estimates)
    return {"std":     round(float(arr.std()), 5),
            "ci_low":  round(float(np.percentile(arr, 2.5)), 5),
            "ci_high": round(float(np.percentile(arr, 97.5)), 5)}


def vol_TIN(pts, sor_k=16, sor_std=2.0, cluster_eps=0.15,
            floor_pct=0.5, run_bootstrap=True, n_bootstrap=80):
    """সম্পূর্ণ volume pipeline: SOR → largest cluster → TIN [→ bootstrap CI].
    Returns (volume, confidence_dict_or_None). একক = pts-এর এককের ঘন (m³)।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 10:
        return float("nan"), None

    pts, _ = sor_filter(pts, k=sor_k, std_ratio=sor_std)          # 1. SOR
    if len(pts) < 10:
        return float("nan"), None

    pts = extract_main_cluster(pts, eps=cluster_eps)               # 2. cluster
    if len(pts) < 10:
        return float("nan"), None

    volume = tin_integrate(pts, floor_pct=floor_pct)               # 3. TIN

    conf = None                                                    # 4. CI
    if run_bootstrap and len(pts) >= 20:
        conf = bootstrap_tin_confidence(pts, n_bootstrap=n_bootstrap,
                                        floor_pct=floor_pct)
    return float(volume), conf


## Testing  (held-out test files with labels)

In [ ]:
# ── Testing on held-out test files ───────────────────────────────────────────
log.info("=== TESTING ===")
test_mious = []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(path)
    miou = compute_miou(lbl, preds, NUM_CLASSES)
    test_mious.append(miou)
    log.info(f"  {name}: mIoU = {miou:.4f}")

log.info(f"Mean test mIoU: {np.mean(test_mious):.4f}")

# ── Visualize + volume for the first few test files ──────────────────────────
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud(path)

    # segmentation → vol_TIN → volume
    target_pts   = pts[preds == CONFIG["target_class"]]
    volume, conf = vol_TIN(target_pts)

    n_tot = len(pts)
    n_tgt = len(target_pts)
    print(f"\n{'─'*55}")
    print(f"  File          : {name}")
    print(f"  Total Points  : {n_tot:,}")
    print(f"  Target Points : {n_tgt:,}")
    print(f"  Other Points  : {n_tot - n_tgt:,}")
    print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
    print(f"  Est. Volume   : {volume:.4f} m³")
    if conf:
        print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
              f"  (±{conf['std']:.4f})")
    print(f"{'─'*55}")

    visualize_segmentation(pts, preds,
                           title=f"{MODEL_NAME} | {name} | V={volume:.4f} m³")


2026-07-10 05:32:10,341 | INFO | === TESTING ===
2026-07-10 05:32:10,404 | INFO |   [plane] wall: 2,314 pts (|nz|=0.02, z̄=78.90)
2026-07-10 05:32:10,549 | INFO |   sample_data_0013: mIoU = 0.5906
2026-07-10 05:32:10,622 | INFO |   [plane] wall: 5,394 pts (|nz|=0.02, z̄=77.88)
2026-07-10 05:32:10,859 | INFO |   sample_data_0020: mIoU = 0.5719
2026-07-10 05:32:10,925 | INFO |   [plane] wall: 6,521 pts (|nz|=0.01, z̄=73.29)
2026-07-10 05:32:11,204 | INFO |   sample_data_0017: mIoU = 0.6775
2026-07-10 05:32:11,238 | INFO |   [plane] wall: 4,557 pts (|nz|=0.02, z̄=72.11)
2026-07-10 05:32:11,251 | INFO |   [plane] wall: 4,449 pts (|nz|=0.02, z̄=72.48)
2026-07-10 05:32:11,451 | INFO |   sample_data_0012: mIoU = 0.8217
2026-07-10 05:32:11,506 | INFO |   [plane] wall: 3,779 pts (|nz|=0.01, z̄=78.20)
2026-07-10 05:32:11,679 | INFO |   sample_data_0015: mIoU = 0.6287
2026-07-10 05:32:11,713 | INFO |   [plane] ground: 2,100 pts (|nz|=1.00, z̄=72.28)
2026-07-10 05:32:11,819 | INFO |   sample_data_

## Final Inference  (`data/test`, no labels)

In [ ]:
# ── Final inference on data/test (no labels needed) ──────────────────────────
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, _ = predict_full_cloud(path)

        # segmentation → vol_TIN → volume
        target_pts   = pts[preds == CONFIG["target_class"]]
        volume, conf = vol_TIN(target_pts)

        n_tot = len(pts)
        n_tgt = len(target_pts)
        print(f"\n{'─'*55}")
        print(f"  File          : {name}")
        print(f"  Total Points  : {n_tot:,}")
        print(f"  Target Points : {n_tgt:,}")
        print(f"  Other Points  : {n_tot - n_tgt:,}")
        print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
        print(f"  Est. Volume   : {volume:.4f} m³")
        if conf:
            print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
                  f"  (±{conf['std']:.4f})")
        print(f"{'─'*55}")

        visualize_segmentation(pts, preds,
                               title=f"INFERENCE | {MODEL_NAME} | {name} | "
                                     f"V={volume:.4f} m³")
else:
    log.info("data/test is empty — skipping inference.")
